In [ ]:
# scripts/12_learning_curve_and_bootstrap.py

"""
Step 12 - Robustness analysis: learning curve + bootstrap (using the precomputed distance matrix).

Inputs:
- outputs/data/functional_space.parquet
- outputs/models/functional_knn_params.pkl
- outputs/matrices/functional_distance_matrix.npy
- outputs/matrices/functional_distance_matrix_nits.npy
- outputs/results/step11_results.pkl   (for the 100% F1 reference)

Outputs:
- outputs/figures/step12_learning_curve.png
- outputs/reports/step12_learning_curve.csv
- outputs/figures/step12_bootstrap_boxplot.png
- outputs/results/step12_bootstrap_metrics.pkl
- outputs/reports/step12_summary.tex
"""

from __future__ import annotations

import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, log_loss
)


# ---------------------------------------------------------------------
# Step 1: Paths
# ---------------------------------------------------------------------
SPACEF_PATH = Path("outputs") / "data" / "functional_space.parquet"
PARAMS_PATH = Path("outputs") / "models" / "functional_knn_params.pkl"
STEP11_RESULTS_PATH = Path("outputs") / "results" / "step11_results.pkl"
DIST_MATRIX_PATH = Path("outputs") / "matrices" / "functional_distance_matrix.npy"
NITS_ORDER_PATH = Path("outputs") / "matrices" / "functional_distance_matrix_nits.npy"

FIG_LEARNING_PATH = Path("outputs") / "figures" / "step12_learning_curve.png"
CSV_LEARNING_PATH = Path("outputs") / "reports" / "step12_learning_curve.csv"
FIG_BOOTSTRAP_PATH = Path("outputs") / "figures" / "step12_bootstrap_boxplot.png"
BOOTSTRAP_PKL_PATH = Path("outputs") / "results" / "step12_bootstrap_metrics.pkl"
SUMMARY_TEX_PATH = Path("outputs") / "reports" / "step12_summary.tex"

for p in [FIG_LEARNING_PATH, CSV_LEARNING_PATH, FIG_BOOTSTRAP_PATH, BOOTSTRAP_PKL_PATH, SUMMARY_TEX_PATH]:
    p.parent.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------
# Step 2: Load inputs
# ---------------------------------------------------------------------
df = pd.read_parquet(SPACEF_PATH)

with open(PARAMS_PATH, "rb") as f:
    params = pickle.load(f)

with open(STEP11_RESULTS_PATH, "rb") as f:
    step11 = pickle.load(f)
step11_summary = step11["summary"]

dist_matrix = np.load(DIST_MATRIX_PATH)
nits_order = np.load(NITS_ORDER_PATH, allow_pickle=True)

k_final = int(params["k"])
lambda_final = float(params["lambda"])

# Optional: extra weights (for reporting only; matrix already includes them)
w_dep = params.get("weight_DEP", params.get("peso_DEP", None))
w_ciiu = params.get("weight_CIIU", params.get("peso_CIIU", None))
w_year = params.get("weight_year", params.get("peso_desfase", None))
max_year_gap = float(params.get("max_year_gap", 24.0))

w_dep = float(w_dep) if w_dep is not None else 0.0
w_ciiu = float(w_ciiu) if w_ciiu is not None else 0.0
w_year = float(w_year) if w_year is not None else 0.0

# Reference (100%) from Step 11 summary stores strings like "0.9147 ±0.0042"
f1_100 = float(str(step11_summary["F1-score"]).split(" ±")[0])
std_100 = float(str(step11_summary["F1-score"]).split(" ±")[1])

# Align df to the distance matrix ordering
df = df[df["RQ_final"].notna()].copy()
if "NIT" in df.columns:
    df = df.set_index("NIT", drop=False)

df = df.loc[nits_order]
y = df["RQ_final"].values.astype(int)
n = len(df)

print("\n[STEP 12] Learning curve + bootstrap robustness analysis")
print(f"[INFO] n={n:,} | k={k_final} | lambda={lambda_final:.4f} | F1(100%)={f1_100:.4f} ±{std_100:.4f}")
print(f"[INFO] Extra metric weights (reported): w_DEP={w_dep:.4f}, w_CIIU={w_ciiu:.4f}, w_year={w_year:.4f} (max_year_gap={max_year_gap:.1f})")


# ---------------------------------------------------------------------
# Step 3: Helper - evaluate model on a submatrix (5-fold CV, mean F1)
# ---------------------------------------------------------------------
def evaluate_functional_knn_with_matrix(matrix_sub: np.ndarray, y_sub: np.ndarray, k: int) -> float:
    """
    Evaluate functional k-NN using a distance submatrix and labels y_sub.
    Uses 5-fold stratified CV and returns mean F1.
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    f1_scores = []

    for train_idx, test_idx in skf.split(matrix_sub, y_sub):
        preds = []
        y_test = y_sub[test_idx]

        for i in test_idx:
            # neighbors among train indices (labels are y_sub[j], not y_train remapping)
            dists = [(matrix_sub[i, j], y_sub[j]) for j in train_idx]
            neighbors = sorted(dists, key=lambda x: x[0])[:k]
            proba = float(np.mean([v[1] for v in neighbors]))
            preds.append(int(round(proba)))

        f1_scores.append(f1_score(y_test, preds))

    return float(np.mean(f1_scores))


# ---------------------------------------------------------------------
# Step 4: Learning curve (reuse submatrices from the full matrix)
# ---------------------------------------------------------------------
percentages = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30, 0.50, 0.70]
repetitions = 3

curve_perc, curve_f1, curve_std = [], [], []

# Map NIT -> global index in distance matrix
nit_to_pos = {nit: i for i, nit in enumerate(nits_order)}

for p in tqdm(percentages, desc="Learning curve"):
    f1_rep = []
    for r in range(repetitions):
        seed = 1000 + r
        sampled_nits = df.sample(frac=p, random_state=seed).index
        idx = [nit_to_pos[nit] for nit in sampled_nits]

        matrix_sub = dist_matrix[np.ix_(idx, idx)]
        y_sub = df.loc[sampled_nits]["RQ_final"].values.astype(int)

        f1_val = evaluate_functional_knn_with_matrix(matrix_sub, y_sub, k_final)
        f1_rep.append(f1_val)

    curve_perc.append(int(p * 100))
    curve_f1.append(float(np.mean(f1_rep)))
    curve_std.append(float(np.std(f1_rep)))

# Add 100% using Step 11 reference (no recomputation)
curve_perc.append(100)
curve_f1.append(f1_100)
curve_std.append(std_100)

# Sort by sample percent (just in case)
order = np.argsort(curve_perc)
curve_perc = [curve_perc[i] for i in order]
curve_f1 = [curve_f1[i] for i in order]
curve_std = [curve_std[i] for i in order]

# Plot learning curve
plt.figure(figsize=(8, 5))
plt.plot(curve_perc, curve_f1, marker="o", label="Mean F1-score")
plt.fill_between(
    curve_perc,
    np.array(curve_f1) - np.array(curve_std),
    np.array(curve_f1) + np.array(curve_std),
    alpha=0.2,
    label="±1 standard deviation",
)
plt.xlabel("Sample size used (%)")
plt.ylabel("F1-score")
plt.title("Learning curve - Functional k-NN model")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIG_LEARNING_PATH, dpi=200)
plt.close()

print(f"[SAVED] Learning curve figure: {FIG_LEARNING_PATH}")

# Save learning curve CSV
pd.DataFrame(
    {
        "sample_percent": curve_perc,
        "f1_mean": curve_f1,
        "f1_std": curve_std,
    }
).to_csv(CSV_LEARNING_PATH, index=False)

print(f"[SAVED] Learning curve CSV: {CSV_LEARNING_PATH}")


# ---------------------------------------------------------------------
# Step 5: Bootstrap robustness (B replications)
# ---------------------------------------------------------------------
B = 30
rng = np.random.default_rng(2025)

metrics = {m: [] for m in ["accuracy", "precision", "recall", "f1", "auc", "logloss"]}

for _ in tqdm(range(B), desc="Bootstrap"):
    train_idx = rng.choice(n, size=n, replace=True)
    test_idx = np.setdiff1d(np.arange(n), train_idx)

    if len(test_idx) == 0:
        continue

    # distances from each test point to each sampled (train) point (with duplicates allowed)
    dists_subset = dist_matrix[np.ix_(test_idx, train_idx)]
    y_train = y[train_idx]
    y_test = y[test_idx]

    preds, probs = [], []
    for row_dists in dists_subset:
        neighbors = np.argsort(row_dists)[:k_final]
        prob = float(np.mean(y_train[neighbors]))
        preds.append(int(round(prob)))
        probs.append(prob)

    metrics["accuracy"].append(accuracy_score(y_test, preds))
    metrics["precision"].append(precision_score(y_test, preds, zero_division=0))
    metrics["recall"].append(recall_score(y_test, preds))
    metrics["f1"].append(f1_score(y_test, preds))
    metrics["auc"].append(roc_auc_score(y_test, probs))
    metrics["logloss"].append(log_loss(y_test, probs))

# Save bootstrap metrics
with open(BOOTSTRAP_PKL_PATH, "wb") as f:
    pickle.dump(metrics, f)

print(f"[SAVED] Bootstrap metrics pickle: {BOOTSTRAP_PKL_PATH}")


# Bootstrap boxplot (accuracy, precision, recall, f1)
df_metrics = pd.DataFrame(metrics)
df_long = df_metrics[["accuracy", "precision", "recall", "f1"]].melt(
    var_name="metric", value_name="value"
)

plt.figure(figsize=(8, 6))
sns.boxplot(data=df_long, x="metric", y="value", palette="pastel")
plt.title(f"Bootstrap metric distribution (B = {B})")
plt.ylim(0.5, 1.0)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(FIG_BOOTSTRAP_PATH, dpi=200)
plt.close()

print(f"[SAVED] Bootstrap boxplot: {FIG_BOOTSTRAP_PATH}")


# ---------------------------------------------------------------------
# Step 6: Save LaTeX summary (English)
# ---------------------------------------------------------------------
bootstrap_summary = pd.DataFrame(metrics).describe().T[["mean", "std"]]

with open(SUMMARY_TEX_PATH, "w", encoding="utf-8") as f:
    f.write(r"\section*{Step 12: Robustness Analysis (Learning Curve and Bootstrap)}" + "\n")
    f.write(r"\begin{itemize}" + "\n")
    f.write(r"  \item A learning curve was computed from 1\% up to 100\% of the sample.\n")
    f.write(f"  \item F1-score at 100\% of the data (Step 11 reference): {f1_100:.4f} $\\pm$ {std_100:.4f}\n")
    f.write(f"  \item A bootstrap procedure with {B} replications was applied to assess stability.\n")
    f.write(r"  \item The robustness analysis leveraged the precomputed distance matrix (extended metric).\n")
    f.write(f"  \item Extra metric weights (for documentation): DEP={w_dep:.4f}, CIIU={w_ciiu:.4f}, year={w_year:.4f} (max year gap={max_year_gap:.1f}).\n")
    f.write(r"  \item Outputs generated:\n")
    f.write(r"  \begin{itemize}" + "\n")
    f.write(r"    \item \texttt{outputs/figures/step12\_learning\_curve.png}\n")
    f.write(r"    \item \texttt{outputs/reports/step12\_learning\_curve.csv}\n")
    f.write(r"    \item \texttt{outputs/figures/step12\_bootstrap\_boxplot.png}\n")
    f.write(r"    \item \texttt{outputs/results/step12\_bootstrap\_metrics.pkl}\n")
    f.write(r"  \end{itemize}" + "\n")
    f.write(r"\end{itemize}" + "\n")

print(f"[SAVED] LaTeX summary: {SUMMARY_TEX_PATH}")

print("\n[BOOTSTRAP SUMMARY] (mean ± std)")
print(bootstrap_summary)